### **Generation: Stuffing Documents**

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [4]:
vectorstore = Chroma(persist_directory="./vector_db",
                     embedding_function=OpenAIEmbeddings(model='text-embedding-ada-002'))

C:\Users\kubte\AppData\Local\Temp\ipykernel_21520\2098445511.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory="./vector_db",


In [5]:
len(vectorstore.get()['documents'])

44

In [6]:
retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.7})

In [8]:
TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [9]:
chat = ChatOpenAI(
    model_name = 'gpt-4o',
    seed = 7,
    max_completion_tokens = 250
)

In [10]:
question = "What software do data scientists use?"

In [17]:
chain = {'context': retriever,
         'question': RunnablePassthrough()} | prompt_template | chat

In [18]:
chain.invoke(question)

AIMessage(content='Data scientists commonly use software such as R and Python due to their adaptability, allowing them to handle a wide range of business and data-related problems from start to finish. Additionally, big data frameworks like Hadoop are crucial for managing complex and computationally intense data. For business intelligence and visualization, tools like Power BI, SaS, Qlik, and Tableau are also frequently employed.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 379, 'total_tokens': 471, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', '

In [20]:
print('Data scientists commonly use software such as R and Python due to their adaptability, allowing them to handle a wide range of business and data-related problems from start to finish. Additionally, big data frameworks like Hadoop are crucial for managing complex and computationally intense data. For business intelligence and visualization, tools like Power BI, SaS, Qlik, and Tableau are also frequently employed.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need')

Data scientists commonly use software such as R and Python due to their adaptability, allowing them to handle a wide range of business and data-related problems from start to finish. Additionally, big data frameworks like Hadoop are crucial for managing complex and computationally intense data. For business intelligence and visualization, tools like Power BI, SaS, Qlik, and Tableau are also frequently employed.

Resources: Programming Languages & Software Employed in Data Science - All the Tools You Need
